In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib as mpl

import re
import calendar
import matplotlib.pyplot as plt
import geopandas as gpd


## Checking Outputs

### Groundwater Basins


In [ ]:
basin_shp_fname = glob.glob('shapefile_gw_basin_all_lands/ensemble_mean.shp')[0]
basin_gdf = gpd.read_file(basin_shp_fname)
basin_gdf.plot(column='ENSavg0310')

In [ ]:

gdf = basin_gdf.copy()
# Example:
# gdf = gpd.read_file("your_file.shp")

model = "ENS"
stat = "avg"

# Match columns like ENSavg2508, ENSavg2304, etc.
pattern = re.compile(rf"^{model}{stat}(\d{{2}})(\d{{2}})$")

# Create dictionary to store columns by month
month_cols = {m: [] for m in range(1, 13)}

for col in gdf.columns:
    match = pattern.match(col)
    if match:
        year = int(match.group(1)) + 2000
        month = int(match.group(2))

        if 1 <= month <= 12:
            month_cols[month].append(col)

# Calculate average ET for each month across all years
for month, cols in month_cols.items():
    if cols:
        gdf[f"ET_month_{month:02d}"] = gdf[cols].mean(axis=1)
    else:
        gdf[f"ET_month_{month:02d}"] = None

# Get shared color scale across all 12 monthly averages
monthly_avg_cols = [f"ET_month_{m:02d}" for m in range(1, 13)]

vmin = gdf[monthly_avg_cols].min().min()
vmax = gdf[monthly_avg_cols].max().max()

# Create 12-panel plot: 4 rows x 3 columns
fig, axes = plt.subplots(
    nrows=4,
    ncols=3,
    figsize=(15, 18),
    constrained_layout=True
)

axes = axes.flatten()

for i, month in enumerate(range(1, 13)):
    ax = axes[i]

    col = f"ET_month_{month:02d}"
    month_name = calendar.month_name[month]

    gdf.plot(
        column=col,
        ax=ax,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        legend=False,
        edgecolor="black",
        linewidth=0.2
    )

    ax.set_title(month_name, fontsize=14)
    ax.set_axis_off()

# Add one shared colorbar
sm = plt.cm.ScalarMappable(
    cmap="viridis",
    norm=plt.Normalize(vmin=vmin, vmax=vmax)
)
sm._A = []

cbar = fig.colorbar(
    sm,
    ax=axes,
    orientation="vertical",
    fraction=0.025,
    pad=0.02
)
cbar.set_label("Average Monthly ET", fontsize=12)

fig.suptitle(
    "Average Monthly Evapotranspiration Across Years",
    fontsize=18,
    y=1.02
)

plt.show()

In [ ]:
# aggregation_unit = 'gw_basin' #counties
basin_shp_fname = glob.glob('shapefile_gw_basin_ag_lands/ensemble_vol.shp')[0]
# basin_shp_fname = glob.glob('shapefile_gw_basin_ag_lands/ensemble_vol.shp')[0]

gdf = gpd.read_file(basin_shp_fname)

model = "ENS"
stat = "vol"

pattern = re.compile(rf"^{model}{stat}(\d{{2}})(\d{{2}})$")

# Store matching ET columns by month
month_cols = {m: [] for m in range(1, 13)}

for col in gdf.columns:
    match = pattern.match(col)
    if match:
        month = int(match.group(2))
        if 1 <= month <= 12:
            month_cols[month].append(col)

# Calculate monthly averages across all years
for month, cols in month_cols.items():
    if cols:
        gdf[f"ET_month_{month:02d}"] = gdf[cols].mean(axis=1)
    else:
        gdf[f"ET_month_{month:02d}"] = float("nan")

# Define seasons in desired subplot order:
# top-left DJF, top-right MAM, bottom-left JJA, bottom-right SON
seasons = {
    "DJF\nDec–Jan–Feb": [12, 1, 2],
    "MAM\nMar–Apr–May": [3, 4, 5],
    "JJA\nJun–Jul–Aug": [6, 7, 8],
    "SON\nSep–Oct–Nov": [9, 10, 11],
}

# Calculate seasonal totals
for season_name, months in seasons.items():
    season_code = season_name.split("\n")[0]
    cols = [f"ET_month_{m:02d}" for m in months]
    gdf[f"ET_{season_code}_tot"] = gdf[cols].sum(axis=1)

season_plot_cols = {
    "Dec–Jan–Feb": "ET_DJF_tot",
    "Mar–Apr–May": "ET_MAM_tot",
    "Jun–Jul–Aug": "ET_JJA_tot",
    "Sep–Oct–Nov": "ET_SON_tot",
}

vmin = gdf[list(season_plot_cols.values())].min().min()
vmax = gdf[list(season_plot_cols.values())].max().max()-1.

fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(14, 12),
    constrained_layout=True
)

axes = axes.flatten()

for ax, (season_name, col) in zip(axes, season_plot_cols.items()):
    gdf.plot(
        column=col,
        ax=ax,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        legend=False,
        edgecolor="black",
        linewidth=0.2
    )

    ax.set_title(season_name, fontsize=14)
    ax.set_axis_off()

sm = plt.cm.ScalarMappable(
    cmap="viridis",
    norm=plt.Normalize(vmin=vmin, vmax=vmax)
)
sm._A = []

cbar = fig.colorbar(
    sm,
    ax=axes,
    orientation="vertical",
    fraction=0.035,
    pad=0.02
)
cbar.set_label("Seasonal Total ET (acre-ft)", fontsize=12)

fig.suptitle(
    "Seasonal Total Evapotranspiration",
    fontsize=18,
    y=1.02
)

plt.show()

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

basin_shp_fname = glob.glob('shapefile_gw_basin_ag_lands/ensemble_vol.shp')[0]
# basin_shp_fname = glob.glob('shapefile_gw_basin_ag_lands/ensemble_vol.shp')[0]

gdf = gpd.read_file(basin_shp_fname)

model = "ENS"
stat = "vol"

# Match columns like ENSavg2508, ENSavg2304, etc.
pattern = re.compile(rf"^{model}{stat}(\d{{2}})(\d{{2}})$")

# Convert wide monthly columns into a long table
records = []

for col in gdf.columns:
    match = pattern.match(col)
    if match:
        yy = int(match.group(1))
        mm = int(match.group(2))

        year = 2000 + yy

        if 2004 <= year <= 2025 and 1 <= mm <= 12:
            records.append({
                "column": col,
                "year": year,
                "month": mm
            })

month_df = pd.DataFrame(records)

# Define seasons
season_map = {
    12: "DJF",
    1: "DJF",
    2: "DJF",
    3: "MAM",
    4: "MAM",
    5: "MAM",
    6: "JJA",
    7: "JJA",
    8: "JJA",
    9: "SON",
    10: "SON",
    11: "SON",
}

month_df["season"] = month_df["month"].map(season_map)

# Use water-year style DJF:
# December belongs to the following winter year.
# Example: Dec 2004 + Jan 2005 + Feb 2005 = DJF 2005
month_df["season_year"] = month_df["year"]

month_df.loc[
    (month_df["month"] == 12) & (month_df["season"] == "DJF"),
    "season_year"
] += 1

# Keep complete seasonal years from 2004 through 2024
month_df = month_df[
    (month_df["season_year"] >= 2005) &
    (month_df["season_year"] <= 2025)
]

# Calculate seasonal ET totals for each polygon and each season-year
seasonal_results = []

for (season_year, season), group in month_df.groupby(["season_year", "season"]):
    cols = group["column"].tolist()

    # Require all 3 months to be present
    if len(cols) == 3:
        seasonal_total = gdf[cols].sum(axis=1)

        seasonal_results.append({
            "year": season_year,
            "season": season,
            "mean_ET": seasonal_total.mean(),
            "total_ET": seasonal_total.sum(),
            "std_ET": seasonal_total.std()
        })

seasonal_ts = pd.DataFrame(seasonal_results)

# Desired panel order:
# top-left DJF, top-right MAM, bottom-left JJA, bottom-right SON
season_order = ["DJF", "MAM", "JJA", "SON"]

fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(14, 10),
    sharex=True
)

axes = axes.flatten()

for ax, season in zip(axes, season_order):
    data = seasonal_ts[seasonal_ts["season"] == season].sort_values("year")

    x = data["year"].values
    y = data["total_ET"].values

    ax.plot(
        x,
        y,
        marker="o",
        linewidth=2,
        label="Seasonal ET"
    )

    # Add linear trend line
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    trend = intercept + slope * x

    ax.plot(
        x,
        trend,
        linestyle="--",
        linewidth=2,
        label="Linear trend"
    )

    ax.set_title(
        f"{season} trend\n"
        f"slope = {slope:.3f} ET units/year, p = {p_value:.3f}",
        fontsize=13
    )

    ax.set_ylabel("Seasonal Total ET")
    ax.grid(True, alpha=0.3)
    ax.legend()

for ax in axes[-2:]:
    ax.set_xlabel("Year")

fig.suptitle(
    "Seasonal ET Trends, 2004–2025",
    fontsize=18,
    y=1.02
)

plt.tight_layout()
plt.show()

In [ ]:
seasonal_ts.mean_ET.plot()


In [ ]:
# aggregation_unit = 'gw_basin' #counties
basin_shp_fname = glob.glob('shapefile_gw_basin_ag_lands/ensemble_mean.shp')[0]
# basin_shp_fname = glob.glob('shapefile_gw_basin_ag_lands/ensemble_vol.shp')[0]

gdf = gpd.read_file(basin_shp_fname)

model = "ENS"
stat = "avg"

pattern = re.compile(rf"^{model}{stat}(\d{{2}})(\d{{2}})$")

# Store matching ET columns by month
month_cols = {m: [] for m in range(1, 13)}

for col in gdf.columns:
    match = pattern.match(col)
    if match:
        month = int(match.group(2))
        if 1 <= month <= 12:
            month_cols[month].append(col)

# Calculate monthly averages across all years
for month, cols in month_cols.items():
    if cols:
        gdf[f"ET_month_{month:02d}"] = gdf[cols].mean(axis=1)
    else:
        gdf[f"ET_month_{month:02d}"] = float("nan")

# Define seasons in desired subplot order:
# top-left DJF, top-right MAM, bottom-left JJA, bottom-right SON
seasons = {
    "DJF\nDec–Jan–Feb": [12, 1, 2],
    "MAM\nMar–Apr–May": [3, 4, 5],
    "JJA\nJun–Jul–Aug": [6, 7, 8],
    "SON\nSep–Oct–Nov": [9, 10, 11],
}

# Calculate seasonal totals
for season_name, months in seasons.items():
    season_code = season_name.split("\n")[0]
    cols = [f"ET_month_{m:02d}" for m in months]
    gdf[f"ET_{season_code}_avg"] = gdf[cols].mean(axis=1)

season_plot_cols = {
    "Dec–Jan–Feb": "ET_DJF_avg",
    "Mar–Apr–May": "ET_MAM_avg",
    "Jun–Jul–Aug": "ET_JJA_avg",
    "Sep–Oct–Nov": "ET_SON_avg",
}

vmin = gdf[list(season_plot_cols.values())].min().min()
vmax = gdf[list(season_plot_cols.values())].max().max()-1.

fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(14, 12),
    constrained_layout=True
)

axes = axes.flatten()

for ax, (season_name, col) in zip(axes, season_plot_cols.items()):
    gdf.plot(
        column=col,
        ax=ax,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        legend=False,
        edgecolor="black",
        linewidth=0.2
    )

    ax.set_title(season_name, fontsize=14)
    ax.set_axis_off()

sm = plt.cm.ScalarMappable(
    cmap="viridis",
    norm=plt.Normalize(vmin=vmin, vmax=vmax)
)
sm._A = []

cbar = fig.colorbar(
    sm,
    ax=axes,
    orientation="vertical",
    fraction=0.035,
    pad=0.02
)
cbar.set_label("Seasonal Avg ET (in)", fontsize=12)

fig.suptitle(
    "Seasonal Average Evapotranspiration",
    fontsize=18,
    y=1.02
)

plt.show()

### Hydrologic Regions

#### plotting functions

In [ ]:
def create_seasonal_plot(seasonal_avg, reference_shp, model_name):
    fig, axs = plt.subplots(2,2, figsize=(8,10))
    axs = axs.flatten()
    # Shared color scale across all seasons
    vmin = 0
    vmax = 6
    i=0
    fig.subplots_adjust(
    left=0.05,
    right=0.95,
    top=0.92,
    bottom=0.12,
    wspace=0.05,
    hspace=0.15
    )
    for season_i in seasonal_avg.season.unique():
        print(model_name,season_i, 'complete')
        season_df_i = seasonal_avg.loc[seasonal_avg.season==season_i].copy()
        seasonal_gdf = reference_shp.merge(
            season_df_i,
            on="HR_NAME",
            how="left"
        )

        seasonal_gdf = gpd.GeoDataFrame(
            seasonal_gdf,
            geometry="geometry",
            crs=reference_shp.crs
        )

        seasonal_gdf.plot(
            column="mean_ET_in",
            ax=axs[i],
            legend=False,#turning off individual colorbars
            edgecolor="black",
            vmin=0,
            vmax=5.5,
            cmap='viridis',
            legend_kwds={"label": "ET [in]"},
            missing_kwds={
                "color": "lightgrey",
                "label": "No data"
            }
        )

        axs[i].set_title(str(season_i))
        axs[i].axis("off")


        i+=1
    # Shared colorbar
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=axs,
        orientation="horizontal",
        fraction=0.04,
        pad=0.04
    )

    cbar.set_label("ET [in]")

    fig.suptitle(
        f"{model_name} Seasonal Average ET by Hydrologic Region",
        fontsize=16,
        y=0.98
    )

    plt.show()

#### plot data

In [ ]:
reference_shp = gpd.read_file('i03_Hydrologic_Regions/i03_Hydrologic_Regions.shp')
reference_shp = reference_shp[reference_shp['OBJECTID'] != 12]
reference_shp = reference_shp[reference_shp['OBJECTID'] != 20]
all_data_df = pd.read_csv("open_data_csvs/regions_ag_mask_open_data.csv")
wy_df = all_data_df.loc[all_data_df.timestep=='water_year'].copy()
monthly_df = all_data_df.loc[all_data_df.timestep=='month'].copy()
monthly_df['DATE']=pd.to_datetime(monthly_df['year_month'])
monthly_df['season_year']=monthly_df.year.astype(int)
monthly_df.loc[monthly_df["DATE"].dt.month == 12, "season_year"] += 1


In [ ]:
models = ['ENSEMBLE','SIMS','DISALEXI','PTJPL','EEMETRIC','SSEBOP','GEESEBAL']

for model_name in models:

    season_map = {
        12: "DJF", 1: "DJF", 2: "DJF",
        3: "MAM", 4: "MAM", 5: "MAM",
        6: "JJA", 7: "JJA", 8: "JJA",
        9: "SON", 10: "SON", 11: "SON",
    }

    monthly_df["season"] = monthly_df["DATE"].dt.month.map(season_map)
    seasonal_totals = (
        monthly_df.groupby(["HR_NAME", "season_year", "season"], as_index=False)
          [f'{model_name}_ET_mean_in']
          .mean()
    )

    seasonal_avg = (
        monthly_df.groupby(["HR_NAME", "season"], as_index=False, observed=True)
          [f'{model_name}_ET_mean_in']
          .mean()
          .rename(columns={f'{model_name}_ET_mean_in': "mean_ET_in"})
          .sort_values(["HR_NAME", "season"])
    )
    create_seasonal_plot(seasonal_avg,reference_shp,model_name)

#### Counties

In [ ]:
def create_seasonal_county_plot(seasonal_avg, reference_shp, merge_id, model_name):
    fig, axs = plt.subplots(2,2, figsize=(8,10))
    axs = axs.flatten()
    # Shared color scale across all seasons
    vmin = 0
    vmax = 6
    i=0
    fig.subplots_adjust(
    left=0.05,
    right=0.95,
    top=0.92,
    bottom=0.12,
    wspace=0.05,
    hspace=0.15
    )
    for season_i in seasonal_avg.season.unique():
        print(model_name,season_i, 'complete')
        season_df_i = seasonal_avg.loc[seasonal_avg.season==season_i].copy()
        seasonal_gdf = reference_shp.merge(
            season_df_i,
            on=merge_id,
            how="left"
        )

        seasonal_gdf = gpd.GeoDataFrame(
            seasonal_gdf,
            geometry="geometry",
            crs=reference_shp.crs
        )

        seasonal_gdf.plot(
            column="mean_ET_in",
            ax=axs[i],
            legend=False,#turning off individual colorbars
            edgecolor="black",
            vmin=0,
            vmax=5.5,
            cmap='viridis',
            legend_kwds={"label": "ET [in]"},
            missing_kwds={
                "color": "lightgrey",
                "label": "No data"
            }
        )

        axs[i].set_title(str(season_i))
        axs[i].axis("off")


        i+=1
    # Shared colorbar
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=axs,
        orientation="horizontal",
        fraction=0.04,
        pad=0.04
    )

    cbar.set_label("ET [in]")

    fig.suptitle(
        f"{model_name} Seasonal Average ET by County",
        fontsize=16,
        y=0.98
    )

    plt.show()

In [ ]:
reference_shp = gpd.read_file('ca_counties/CA_Counties.shp')
all_data_df = pd.read_csv("open_data_csvs/county_ag_mask_open_data.csv")
wy_df = all_data_df.loc[all_data_df.timestep=='water_year'].copy()
monthly_df = all_data_df.loc[all_data_df.timestep=='month'].copy()
monthly_df['DATE']=pd.to_datetime(monthly_df['year_month'])
monthly_df['season_year']=monthly_df.year.astype(int)
monthly_df.loc[monthly_df["DATE"].dt.month == 12, "season_year"] += 1


In [ ]:
models = ['ENSEMBLE','SIMS','DISALEXI','PTJPL','EEMETRIC','SSEBOP','GEESEBAL']

for model_name in models:

    season_map = {
        12: "DJF", 1: "DJF", 2: "DJF",
        3: "MAM", 4: "MAM", 5: "MAM",
        6: "JJA", 7: "JJA", 8: "JJA",
        9: "SON", 10: "SON", 11: "SON",
    }

    monthly_df["season"] = monthly_df["DATE"].dt.month.map(season_map)
    seasonal_totals = (
        monthly_df.groupby(["NAME", "season_year", "season"], as_index=False)
          [f'{model_name}_ET_mean_in']
          .mean()
    )

    seasonal_avg = (
        monthly_df.groupby(["NAME", "season"], as_index=False, observed=True)
          [f'{model_name}_ET_mean_in']
          .mean()
          .rename(columns={f'{model_name}_ET_mean_in': "mean_ET_in"})
          .sort_values(["NAME", "season"])
    )
    create_seasonal_county_plot(seasonal_avg,reference_shp,"NAME",model_name)